# Deliverable 5 — Post-selection over realistic trajectories

Measures block normalization and state-dependent success probabilities at multiple times and Reynolds numbers using actual LBM populations.

This notebook is an executable evidence artifact. Its default configuration is
deliberately small enough for a clean local rerun; scale-up parameters are
listed separately and are not represented as measured results.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "deliverables" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
output_dir = repo_root / "results" / "deliverables"
output_dir.mkdir(parents=True, exist_ok=True)

## Why this matters

The proposal's preliminary 1.77% success probability used one local state at
\(\omega=1.2\). Here \(\omega\) is taken from each physical simulation and
probabilities are evaluated over every grid cell at six trajectory snapshots.
The reported product is the naive no-amplification multi-step probability and
is included only to expose compounding—not as the proposed implementation.

In [2]:
import pandas as pd
import numpy as np
from quantum_aero.classical import LBMConfig, run_lbm
from quantum_aero.deliverables import postselection_statistics

rows = []
for reynolds in (10, 100, 400, 1000):
    result = run_lbm(LBMConfig(n=16, reynolds=reynolds, t_end=0.1, mach=0.05, snapshots=6), keep_fields=True)
    for record, field in zip(result["records"], result["fields"]):
        f = np.moveaxis(field["f"], 0, -1)
        stats = postselection_statistics(f, result["omega"])
        rows.append({"reynolds": reynolds, "step": record["step"], "time": record["time"],
                     "total_steps": result["steps"], **stats,
                     "log10_naive_full_trajectory_p_worst": result["steps"] * np.log10(stats["p_min"])})
df = pd.DataFrame(rows)
df.to_csv(output_dir / "05_postselection_trajectory.csv", index=False)
df

,reynolds,step,time,total_steps,omega,alpha,p_min,p_median,p_max,raw_attempts_worst,aa_scale_worst,log10_naive_full_trajectory_p_worst
0,10,0,0.00,20,1.961835,12.342184,0.006565,0.006567,0.006571,152.329397,12.342180,-43.655674
1,10,4,0.02,20,1.961835,12.342184,0.006565,0.006566,0.006568,152.326583,12.342066,-43.655514
2,10,8,0.04,20,1.961835,12.342184,0.006565,0.006566,0.006567,152.328283,12.342134,-43.655611
3,10,12,0.06,20,1.961835,12.342184,0.006565,0.006567,0.006570,152.329249,12.342174,-43.655666
4,10,16,0.08,20,1.961835,12.342184,0.006564,0.006567,0.006572,152.336633,12.342473,-43.656087
5,10,20,0.10,20,1.961835,12.342184,0.006564,0.006568,0.006573,152.336172,12.342454,-43.656061
6,100,0,0.00,20,1.996117,12.563801,0.006335,0.006338,0.006341,157.848969,12.563796,-43.964835
7,100,4,0.02,20,1.996117,12.563801,0.006335,0.006337,0.006338,157.844769,12.563629,-43.964604
8,100,8,0.04,20,1.996117,12.563801,0.006335,0.006336,0.006337,157.846983,12.563717,-43.964726
9,100,12,0.06,20,1.996117,12.563801,0.006335,0.006337,0.006339,157.847060,12.563720,-43.964730


In [3]:
summary = df.groupby("reynolds").agg(
    omega=("omega", "first"), alpha=("alpha", "first"),
    p_min=("p_min", "min"), p_median=("p_median", "median"),
    aa_scale_worst=("aa_scale_worst", "max"),
    log10_naive_full_trajectory_p_worst=("log10_naive_full_trajectory_p_worst", "min"),
).reset_index()
summary.to_csv(output_dir / "05_postselection_summary.csv", index=False)
assert len(df) == 24
print("PASS: probabilities measured for every cell at 24 trajectory/Re checkpoints.")
summary

PASS: probabilities measured for every cell at 24 trajectory/Re checkpoints.


,reynolds,omega,alpha,p_min,p_median,aa_scale_worst,log10_naive_full_trajectory_p_worst
0,10,1.961835,12.342184,0.006564,0.006567,12.342473,-43.656087
1,100,1.996117,12.563801,0.006335,0.006337,12.564302,-43.965534
2,400,1.999028,12.582637,0.006316,0.006318,12.583169,-43.991602
3,1000,1.999611,12.586411,0.006312,0.006315,12.586950,-43.996820
